# Train and diagnose A-Geo / C-Geo
Setup → resolve existing numerical certification → select the sequential diagnostic suite (or an optional single case) → review → execute its launch cell → inspect results. The optional mixed/research workflow is below the diagnostic section.

**Saved default: preview, zero optimizer updates.** Setting a run mode to `train` or `resume` makes its explicit launch cell run synchronously and immediately. **Run All with training mode enabled can train.** No launch button, confirmation prompt or background training thread is used. Numerical qualification never starts automatically.

The sequential diagnostic suite is the normal Phase G multi-experiment entry point; its saved mode is preview. Advanced single-run/comparison/mixed workflows remain optional below.

In [ ]:
# Setup configuration — used only in Colab. Local sessions need an installed project.
REPO_URL = "SET_YOUR_GITHUB_REPOSITORY_URL"
REPO_REF = "main"  # Prefer an immutable full commit SHA or release tag for research.
PROJECT_DIR = "/content/pde-inverse-research"

USE_GOOGLE_DRIVE = False
DRIVE_ROOT = "/content/drive/MyDrive/PDEInverseResearch"
LOCAL_OUTPUT_DIR = "/content/pde-inverse-outputs"

# Geometry installation extra required for G1_smooth_v1.
ENABLE_GEOMETRY = True


In [ ]:
# Automatic setup — advanced implementation (collapsed). Run this cell normally.
# Stdlib bootstrap is embedded because pdeep is absent in a fresh Colab runtime.
import sys, json
from pathlib import Path
try:
    from IPython import get_ipython
    _shell = get_ipython()
except ImportError:
    _shell = None
IS_COLAB = "google.colab" in sys.modules or type(_shell).__module__.startswith("google.colab")

def ensure_checkout(repo_url, repo_ref, project_dir, *, git_env=None):
    """Clone only absent destinations; existing worktrees are strictly read-only.

    Supports token-free HTTPS and Git-over-SSH URLs. External credential helpers
    or SSH agents handle private authentication. Raw Git output stays in memory.
    """
    import os, re, subprocess, tempfile
    from pathlib import Path
    from urllib.parse import urlsplit
    import sys
    if sys.version_info < (3, 10):
        raise RuntimeError('PDEep requires Python >=3.10; select a supported runtime before setup.')
    def identity(value):
        if not isinstance(value,str) or not value or any(c in value for c in '\n\r\0'):
            raise ValueError('Configure a token-free repository URL; credentials must not be embedded.')
        if value.startswith('git@'):
            match=re.fullmatch(r'git@([A-Za-z0-9.-]+):([A-Za-z0-9_./-]+)',value)
            if not match:raise ValueError('Use git@HOST:OWNER/REPO.git with an external SSH agent.')
            host,path=match.groups()
        else:
            url=urlsplit(value)
            if url.scheme not in {'https','ssh'} or not url.hostname or url.password or url.query or url.fragment:
                raise ValueError('Use token-free HTTPS or Git SSH; credential URLs, queries and fragments are rejected.')
            if (url.scheme=='https' and url.username is not None) or (url.scheme=='ssh' and url.username not in {None,'git'}):
                raise ValueError('Do not put credentials/user tokens in repository URLs; use an external credential helper or SSH agent.')
            host=url.hostname;path=url.path.lstrip('/')
            if url.port not in {None,22,443}:raise ValueError('Unsupported repository port.')
        path=path.rstrip('/').removesuffix('.git')
        if not path or len(path.split('/'))<2 or '..' in path.split('/'):
            raise ValueError('Configure REPO_URL with OWNER/REPO; no local or credential-bearing URL is accepted.')
        if host.lower()=='github.com':path=path.lower()
        return host.lower()+'/'+path
    expected=identity(repo_url)
    if not isinstance(repo_ref,str) or not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_./-]*',repo_ref) or '..' in repo_ref:
        raise ValueError('REPO_REF must be a branch, tag or full commit SHA, not a command or revision expression.')
    destination=Path(project_dir).expanduser().resolve()
    env=dict(os.environ);env.update(git_env or {});env['GIT_TERMINAL_PROMPT']='0'
    def git(arguments, *, cwd=None, purpose='repository verification'):
        try:result=subprocess.run(['git',*arguments],cwd=cwd,env=env,capture_output=True,text=True,timeout=180)
        except FileNotFoundError:raise RuntimeError('Git is missing; install Git before repository setup.') from None
        except (OSError,subprocess.TimeoutExpired):raise RuntimeError('Git '+purpose+' could not complete; no credentials/output were logged.') from None
        if result.returncode:
            error=result.stderr.lower()
            if any(s in error for s in ('authentication','could not read username','publickey','permission denied')):
                raise RuntimeError('Git authentication failed. Configure an external credential helper or SSH agent with read access; do not put secrets in this notebook.')
            raise RuntimeError('Git '+purpose+' failed. Check repository existence, access and REPO_REF. Raw output is suppressed to protect credentials.')
        return result.stdout.strip()
    def head(path):return git(['rev-parse','--verify','HEAD^{commit}'],cwd=path)
    def resolved_ref(path):
        if re.fullmatch(r'[0-9a-fA-F]{40}|[0-9a-fA-F]{64}',repo_ref):wanted=repo_ref.lower()
        else:
            refs=[repo_ref,repo_ref+'^{}'] if repo_ref.startswith('refs/') else ['refs/heads/'+repo_ref,'refs/tags/'+repo_ref,'refs/tags/'+repo_ref+'^{}']
            lines=git(['ls-remote','--exit-code','origin',*refs],cwd=path,purpose='remote reference lookup')
            mapping=dict((ref,sha) for sha,ref in (line.split() for line in lines.splitlines()))
            if repo_ref.startswith('refs/'):wanted=mapping.get(repo_ref+'^{}',mapping.get(repo_ref))
            else:
                branch=mapping.get('refs/heads/'+repo_ref);tag=mapping.get('refs/tags/'+repo_ref+'^{}',mapping.get('refs/tags/'+repo_ref))
                if branch and tag and branch!=tag:raise ValueError('Branch/tag name is ambiguous; use refs/heads/... or refs/tags/....')
                wanted=branch or tag
            if not wanted:raise ValueError('Configured REPO_REF was not found on origin.')
        return wanted
    existed=destination.exists()
    if existed:
        if not destination.is_dir() or not (destination/'.git').exists():
            raise ValueError('PROJECT_DIR exists but is not a Git checkout; choose a separate empty directory. Nothing was overwritten.')
        top=Path(git(['rev-parse','--show-toplevel'],cwd=destination)).resolve()
        if top!=destination:raise ValueError('PROJECT_DIR must be the repository root, not a nested directory.')
        origin=git(['config','--get','remote.origin.url'],cwd=destination)
        if identity(origin)!=expected:raise ValueError('Existing checkout origin differs from REPO_URL. No files or remotes were changed.')
        if git(['status','--porcelain','--untracked-files=normal'],cwd=destination):
            raise ValueError('Existing checkout has local modifications/untracked files. Commit them yourself or choose a new PROJECT_DIR; no reset/checkout was performed.')
        commit=head(destination)
        wanted=resolved_ref(destination)
        if commit!=wanted:
            raise ValueError('Existing checkout is at '+commit+' but REPO_REF resolves to '+wanted+'. Choose a new PROJECT_DIR or update it explicitly; nothing was reset.')
    else:
        destination.parent.mkdir(parents=True,exist_ok=True)
        # A failed clone stays in its own scratch directory, not PROJECT_DIR.
        with tempfile.TemporaryDirectory(prefix='.pdeep-clone-',dir=destination.parent) as scratch:
            checkout=Path(scratch)/'checkout'
            git(['clone','--no-checkout','--',repo_url,str(checkout)],purpose='clone')
            wanted=resolved_ref(checkout)
            git(['fetch','--depth','1','origin',wanted],cwd=checkout,purpose='configured reference fetch')
            git(['checkout','--detach','FETCH_HEAD'],cwd=checkout,purpose='fresh checkout')
            commit=head(checkout)
            if commit!=wanted:raise RuntimeError('Fetched checkout differs from the resolved reference; setup stopped.')
            required=('pyproject.toml','src/pdeep/__init__.py','src/pdeep/colab_setup.py','docs/scientific_contract_v1.md')
            if any(not (checkout/p).is_file() for p in required):
                raise ValueError('Repository/ref lacks the expected PDEep structure or this Colab setup version; select a compatible commit.')
            os.rename(checkout,destination)
    required=('pyproject.toml','src/pdeep/__init__.py','src/pdeep/colab_setup.py','docs/scientific_contract_v1.md')
    if any(not (destination/p).is_file() for p in required):raise ValueError('Expected pyproject.toml, pdeep package/setup and scientific contract are missing.')
    return {'project_dir':str(destination),'repository_identity':expected,'requested_ref':repo_ref,'git_commit':commit,
            'existing_checkout':existed,'worktree_modified_by_setup':not existed,'credentials_recorded':False}

BOOTSTRAP = None
if IS_COLAB:
    print("1. Verifying repository and preparing dependencies before scientific project imports.")
    BOOTSTRAP = ensure_checkout(REPO_URL, REPO_REF, PROJECT_DIR)
    import runpy
    _reference_setup = runpy.run_path(str(Path(BOOTSTRAP["project_dir"]) / "src/pdeep/smooth/bootstrap.py"))
    ENVIRONMENT_SETUP = _reference_setup["prepare_reference_environment"](BOOTSTRAP["project_dir"], is_colab=True)
    _reference_setup["install_project"](BOOTSTRAP["project_dir"], source_revision=BOOTSTRAP["git_commit"])
    print("Dependency preparation complete; no loaded binary was replaced. Existing numerical certification is resolved below; no automatic requalification.")
print("2. Scientific imports and resolved environment.")
from pdeep.colab_setup import finish_setup
SETUP = finish_setup(is_colab=IS_COLAB, bootstrap=BOOTSTRAP, use_drive=USE_GOOGLE_DRIVE,
    drive_root=DRIVE_ROOT, local_output_dir=LOCAL_OUTPUT_DIR)
SETUP_CONFIGURATION = (REPO_URL, REPO_REF, PROJECT_DIR, USE_GOOGLE_DRIVE, DRIVE_ROOT, LOCAL_OUTPUT_DIR)
ROOT = Path(SETUP["project_dir"])
from IPython.display import display, Markdown
from pdeep.notebook_workflows import initialize
from pdeep.runtime import capabilities
print("Colab:", IS_COLAB, "| imported source:", ROOT / "src/pdeep")
print("Exact Git commit:", SETUP["git_commit"] or "unavailable: local project is not a Git checkout")
print("Python:", SETUP["runtime"]["python"])
print("Dependency versions:", json.dumps(SETUP["runtime"]["dependencies"], indent=2))
for c in capabilities(): print(c.device, "available="+str(c.available), c.detail)
if IS_COLAB: print("Validated CUDA:", SETUP["cuda"])
print("Local smooth run root:", Path(SETUP["local_output_dir"]) / "G1_smooth_v1/runs")
print("Drive checkpoints:", None if not SETUP["drive_root"] else Path(SETUP["drive_root"]) / "checkpoints")
print(SETUP["message"])

from pdeep.smooth.bootstrap import environment_versions
print("Python executable:", sys.executable)
print("Installed / loaded dependencies:", json.dumps(environment_versions(), sort_keys=True))


## Existing numerical certification
The issued Linux certificate is stored in the project registry in full, with original-byte SHA256 verification. Search: configured path → local cache → Drive → project registry. Copying it does not qualify an incompatible runtime. The existing qualified local runtime record may be selected explicitly below. Missing/incompatible evidence blocks training; it never launches the 96-case certification.

Google Drive, when enabled in setup, uses `DRIVE_ROOT/certifications/G1_smooth_v1/<certificate_id>.json`. A numerical certificate does not certify neural learning.

In [ ]:
from pdeep.learning_diagnostics.certificates import resolve_certification, copy_certificate, SUPPLIED_ID
CERTIFICATE_ID = SUPPLIED_ID
CERTIFICATE_PATH = None  # Optional full issued JSON path; no upload-path assumption.
USE_EXISTING_LOCAL_QUALIFICATION = not IS_COLAB  # Explicit local policy: reuse existing, never recompute.
CERTIFICATE_CACHE = Path(SETUP["local_output_dir"]) / "certifications/G1_smooth_v1"

ISSUED_CERTIFICATION = resolve_certification("G1_smooth_v1", {"certification_id": CERTIFICATE_ID},
    configured_path=CERTIFICATE_PATH, drive_root=SETUP["drive_root"], cache_dir=CERTIFICATE_CACHE)
display(ISSUED_CERTIFICATION.summary())
CERTIFICATION = ISSUED_CERTIFICATION
if not IS_COLAB and USE_EXISTING_LOCAL_QUALIFICATION and not CERTIFICATION.verified:
    CERTIFICATION = resolve_certification("G1_smooth_v1", {"certification_id": None}, cache_dir=CERTIFICATE_CACHE)
    print("Explicit local policy: selecting an existing native-runtime qualification; issued Linux qualification remains incompatible here.")
    display(CERTIFICATION.summary())
CERTIFICATE = CERTIFICATION.record if CERTIFICATION.verified else None
if CERTIFICATE is None:
    print("Training blocked. Restore the certified runtime/settings or place the full issued JSON in the displayed cache/Drive directory. No requalification has been executed.")
if CERTIFICATION.verified and SETUP["drive_root"]:
    try:
        CERTIFICATE_DRIVE_PATH = copy_certificate(CERTIFICATION.location,
            Path(SETUP["drive_root"]) / "certifications/G1_smooth_v1")
        print("Full certificate byte-copy verified:", CERTIFICATE_DRIVE_PATH)
    except Exception as error:
        print("CERTIFICATE DRIVE COPY FAILED; verified local copy retained:", type(error).__name__, str(error))
PERSISTENCE = SETUP["persistence"]

## Sequential geometry-aware learning diagnostics
The default suite runs **Poisson rectangle → heat square → wave square**, independently initialized, using **8,192 dense resampled query records/update**. These are fixed TRAIN-case capacity diagnostics and independent-coordinate interpolation; new-case generalization is **not tested**. The Poisson-square trained weights are never reused.

Run setup and the certificate resolver above, then the configuration/review below. Set `SUITE_RUN_MODE="train"` and execute the **suite execution cell**. It stays active through all experiments with one dashboard. Saved mode is `preview`: Run All performs no suite optimizer updates. Each run uses its own optimizer/scheduler/checkpoints, the existing numerical certificate, cached CPU64 reference, and configured local-first/Drive persistence. Enabling train also enables training through Run All.

Dynamic v3 diagnostic stratification is per quantity: 12.5% t=0, 12.5% early, 25% middle, 25% late, 25% random continuous [0,1]. Wave totals **4,096 u + 4,096 v**, with equal standardized field contributions. This is an explicit diagnostic query-risk version; old streams/checkpoints and the physical data distribution stay unchanged. C retains its original physics objective.


In [ ]:
DIAGNOSTIC_SUITE = ["poisson_rectangle", "heat_single", "wave_single"]
SUITE_MODEL = "A_GEO"
SUITE_CAPACITY = "medium"
SUITE_QUERY_MODE = "dense_resampled"
SUITE_QUERY_COUNT = 8192
SUITE_UPDATES_PER_EXPERIMENT = 1000
SUITE_LR = 3e-4
SUITE_SCHEDULER = "constant"
SUITE_DEVICE = "cuda" if IS_COLAB else "auto"
SUITE_AMP_MODE = "auto"
SUITE_ACTION = "fresh"
SUITE_RUN_MODE = "preview"
SUITE_SEED = 42
SUITE_QUERY_CHUNK_SIZE = 1024
SUITE_QUERY_CHECKPOINTING = False
SUITE_STOP_ON_FAILURE = True
# Optional resume: select the latest verified suite-state-XXXXXXXX.json AND adjacent .pt.
SUITE_RESUME_MANIFEST = None
SUITE_QUERY_TIME_POLICY = "stratified"
SUITE_QUERY_SAMPLER_SEED = 42
SUITE_ADVANCED = {}  # Optional grid_size/validation_every/checkpoint_every; C ablation pde/family.


In [ ]:
from IPython.display import HTML
import html
from pdeep.learning_diagnostics.suite import prepare_diagnostic_suite, run_diagnostic_suite
from pdeep.learning_diagnostics.suite_reports import display_suite_results

def suite_settings():
    if SETUP_CONFIGURATION != (REPO_URL, REPO_REF, PROJECT_DIR, USE_GOOGLE_DRIVE, DRIVE_ROOT, LOCAL_OUTPUT_DIR):
        raise RuntimeError("Setup changed: rerun setup and suite review.")
    device = "cuda" if IS_COLAB and SUITE_DEVICE == "auto" else SUITE_DEVICE
    if IS_COLAB and not str(device).startswith("cuda"):
        raise RuntimeError("Colab suite requires validated CUDA; no silent CPU fallback.")
    return dict(experiment_names=tuple(DIAGNOSTIC_SUITE), model=SUITE_MODEL, capacity=SUITE_CAPACITY,
        settings=dict(query_mode=SUITE_QUERY_MODE, query_count=SUITE_QUERY_COUNT,
            query_time_policy=SUITE_QUERY_TIME_POLICY, query_sampler_seed=SUITE_QUERY_SAMPLER_SEED,
            updates=SUITE_UPDATES_PER_EXPERIMENT, learning_rate=SUITE_LR, scheduler=SUITE_SCHEDULER,
            device=device, amp_mode=SUITE_AMP_MODE, seed=SUITE_SEED, query_chunk=SUITE_QUERY_CHUNK_SIZE,
            query_checkpointing=SUITE_QUERY_CHECKPOINTING, stop_on_failure=SUITE_STOP_ON_FAILURE,
            advanced=SUITE_ADVANCED.copy()),
        certificate=CERTIFICATE, root=SETUP["local_output_dir"], persistence=PERSISTENCE,
        action=SUITE_ACTION, resume_manifest=SUITE_RESUME_MANIFEST)

SUITE_PLAN = None
SUITE_SETTINGS = suite_settings()
if CERTIFICATION.verified:
    SUITE_PLAN = prepare_diagnostic_suite(**SUITE_SETTINGS)
    SUITE_REVIEWED_IDENTITY = SUITE_PLAN.identity
    display(Markdown("**Suite order:** " + " → ".join(DIAGNOSTIC_SUITE) +
        f" · {SUITE_MODEL}/{SUITE_CAPACITY} · {SUITE_QUERY_COUNT} queries/update · {SUITE_UPDATES_PER_EXPERIMENT} updates each · zero optimizer updates in preview."))
    display(HTML("<details><summary>Resolved suite, identities, losses and time policy</summary><pre>" +
        html.escape(json.dumps(SUITE_PLAN.preview(), indent=2, default=str)) + "</pre></details>"))
else:
    print("Suite blocked by missing/incompatible existing numerical certification; no requalification launched.")


In [ ]:
# Suite execution cell — synchronous direct launch, no button or confirmation.
SUITE_RESULTS = None
if SUITE_RUN_MODE != "preview":
    if suite_settings() != SUITE_SETTINGS:
        raise RuntimeError("Suite settings changed: rerun configuration/review.")
    CERTIFICATION.require()
    if SUITE_PLAN is None:
        raise RuntimeError("Review the suite first.")
    SUITE_RESULTS = run_diagnostic_suite(SUITE_PLAN, expected_identity=SUITE_REVIEWED_IDENTITY,
        run_mode=SUITE_RUN_MODE)
    display_suite_results(SUITE_RESULTS)
else:
    print("Suite preview: zero optimizer updates. Set SUITE_RUN_MODE='train' and run this cell to execute all selected experiments.")
# Resume after disconnect: setup/mount Drive; restore the same settings/commit;
# SUITE_ACTION='resume', SUITE_RUN_MODE='resume', SUITE_RESUME_MANIFEST=<verified suite snapshot>;
# rerun review and this cell. Verified completed members are skipped automatically.
# Existing single-run and optional comparison workflows remain below, at preview.


## Optional single-experiment geometry-aware diagnostics
The physical TRAIN case, source/initial state, sparse noisy sensors, certified reference and independent evaluation grid remain fixed. Only supervised coordinates change in `resampled` and `dense_resampled` modes. `fixed` keeps its coordinate/value set and serves as the memorization control. The original 128-point fixed checkpoint remains readable; exact continuation keeps its old policy and never silently adopts resampling.

Training targets are evaluated from a retained **CPU64 P4** solution/trajectory, with the certified C1 continuous-time output and wave velocity derivative. No PDE solve is repeated per update and no new target enters conditioning. The fixed control and independent scores are separate from the current resampled loss. Dynamic stratification retains the .125 t0/.125 early/.75 uniform[0,1] risk mixture.

Experiments: `poisson_square`, `poisson_rectangle`, `heat_single`, `wave_single`, paired `c_physics_ablation`, read-only `observation_sensitivity`. Architectures/loss weights/priors/observations/certification thresholds remain unchanged. A/C encode equation, geometry and sensors once per update; independent decoder query chunks use exact global quantity/auxiliary reductions. Optional activation checkpointing trades recomputation for memory. CUDA auto-AMP must pass its real FP32/BF16 pilot; derivative-intensive full C stays FP32. Preflight measures one clone's forward/backward, with an explicit optimizer allowance; it does not train.

Low allocation alone is not evidence of poor GPU compute utilization. CUDA allocated/reserved/per-step peak, phase spans and device-wide utilization (when measurable) are reported. CPU reference work and unavoidable geometry point-location are identified separately.

In [ ]:
DIAGNOSTIC_MODEL = "A_GEO"
DIAGNOSTIC_EXPERIMENT = "poisson_square"
DIAGNOSTIC_CAPACITY = "medium"
QUERY_MODE = "dense_resampled"  # fixed, resampled, dense_resampled
QUERY_COUNT = 8192
QUERY_TIME_POLICY = "stratified"
QUERY_SAMPLER_SEED = 42
DIAGNOSTIC_UPDATES = 1000
DIAGNOSTIC_LR = 3e-4
DIAGNOSTIC_DEVICE = "cuda" if IS_COLAB else "auto"  # No silent fallback from explicit CUDA.
DIAGNOSTIC_SEED = 42
DIAGNOSTIC_ACTION = "fresh"
DIAGNOSTIC_RUN_MODE = "preview"
AMP_MODE = "auto"  # Actual CUDA BF16 capability + loss/gradient pilot; otherwise FP32.
MEMORY_TARGET_GB = 11

In [ ]:
# Optional advanced diagnostics — leave collapsed for the normal workflow.
DIAGNOSTIC_CHECKPOINTS = None  # Resume: verified manifest; ablation: {"FULL": path, "FIELD_ONLY": path}.
DIAGNOSTIC_RUN_NAME = None  # Fresh gets a unique tag; existing fresh runs are rejected.
DIAGNOSTIC_ADVANCED = {}  # e.g. {"pde":"wave"} for ablation/sensitivity; constant LR is the explicit diagnostic default.
RUN_DIAGNOSTIC_PREFLIGHT = False  # Forward/coordinate diagnostics only; no optimizer.
DIAGNOSTIC_LOSS_GRADIENTS = False  # More expensive per-term gradient norms/cosines, still zero updates.
INSPECT_CHECKPOINT = None  # Optional old 1000-update Drive manifest; read-only inspection, never a warm start.
# A-Geo: DRIVE_ROOT/checkpoints/A_GEO-9e190bc29a32701f10fefc74/8fb8ee2a78f1f08d337dd7af1bbb369cd853b315449d8884bb4e65f1fada6ea1/smooth-update-00001000.json
# C-Geo: DRIVE_ROOT/checkpoints/C_GEO-db0daa0fe44fa5b68399f1ad/f636e9914051e300747098454be48bdd455e05cf9a648035e35992961d9cdb52/smooth-update-00001000.json
QUERY_CHUNK_SIZE = 1024
QUERY_CHECKPOINTING = False  # Enable if measured memory is too high; no query-to-query attention.
RUN_PERFORMANCE_PREFLIGHT = True  # ONE isolated forward/backward; zero optimizer updates.
PERFORMANCE_QUERY_COUNTS = None  # Optional bounded sweep: (1024,2048,4096,8192); add16384 explicitly.
STRIPING_CHECKPOINT = None  # Verified local/Drive JSON manifest AND its .pt payload must be together.
RUN_STRIPING_AUDIT = False  # Read-only actual trained-weight multi-layout audit.

LEGACY_FIXED_RESUME = False  # Only for original fixed-query checkpoints: also restore QUERY_COUNT=128, seed/budget/device as saved.

PROFILE_CUDA_KERNELS = False  # Optional isolated kernel-active timing; slower profiler, never whole-run estimate.


In [ ]:
from pdeep.learning_diagnostics.workflows import prepare_diagnostic, diagnostic_preview, preflight, launch_diagnostic
from pdeep.learning_diagnostics.dashboard import Dashboard
from IPython.display import HTML
import html

def diagnostic_settings():
    if SETUP_CONFIGURATION != (REPO_URL, REPO_REF, PROJECT_DIR, USE_GOOGLE_DRIVE, DRIVE_ROOT, LOCAL_OUTPUT_DIR):
        raise RuntimeError("Setup changed: rerun setup and preview.")
    device = "cuda" if IS_COLAB and DIAGNOSTIC_DEVICE == "auto" else DIAGNOSTIC_DEVICE
    if IS_COLAB and not str(device).startswith("cuda"):
        raise RuntimeError("Colab diagnostics require validated CUDA; no silent CPU fallback.")
    return dict(model=DIAGNOSTIC_MODEL, experiment=DIAGNOSTIC_EXPERIMENT, capacity=DIAGNOSTIC_CAPACITY,
        updates=DIAGNOSTIC_UPDATES, device=device, seed=DIAGNOSTIC_SEED, action=DIAGNOSTIC_ACTION,
        certificate=CERTIFICATE, root=SETUP["local_output_dir"], persistence=PERSISTENCE,
        checkpoints=DIAGNOSTIC_CHECKPOINTS, run_name=DIAGNOSTIC_RUN_NAME, advanced={**DIAGNOSTIC_ADVANCED, "learning_rate": DIAGNOSTIC_LR},
        query_mode=None if LEGACY_FIXED_RESUME else QUERY_MODE, query_count=QUERY_COUNT, query_time_policy=QUERY_TIME_POLICY,
        query_sampler_seed=QUERY_SAMPLER_SEED, query_chunk=QUERY_CHUNK_SIZE,
        query_checkpointing=QUERY_CHECKPOINTING, amp_mode=AMP_MODE, memory_target_gb=MEMORY_TARGET_GB)

DIAGNOSTIC_SETTINGS = diagnostic_settings()
DIAGNOSTIC_PLAN = None
if CERTIFICATION.verified:
    print("Preparing one fixed TRAIN case and real model (CPU64 reference labels once; zero optimizer updates).")
    DIAGNOSTIC_PLAN = prepare_diagnostic(**DIAGNOSTIC_SETTINGS)
    if RUN_PERFORMANCE_PREFLIGHT:
        from pdeep.learning_diagnostics.workflows import performance_preflight
        PERFORMANCE_REPORT = performance_preflight(DIAGNOSTIC_PLAN, counts=PERFORMANCE_QUERY_COUNTS, cuda_kernel_profile=PROFILE_CUDA_KERNELS)
        for objective, report in PERFORMANCE_REPORT.items():
            display(Markdown(f"**{objective} performance preflight** · precision `{report['precision']}`"))
            for row in report["profiles"]:
                display({k:row.get(k) for k in ("status","query_count","step_wall_seconds","queries_per_second","memory","estimated_peak_with_optimizer_bytes","optimizer_state_scope","phase_CUDA_event_seconds")})
    DIAGNOSTIC_REVIEWED_IDENTITY = DIAGNOSTIC_PLAN.identity
    resolved = diagnostic_preview(DIAGNOSTIC_PLAN)
    display(Markdown(f"**{DIAGNOSTIC_MODEL} · {DIAGNOSTIC_EXPERIMENT}** · parameters {resolved['parameters']} · {DIAGNOSTIC_PLAN.configs[0].device} · fixed control queries {int(DIAGNOSTIC_PLAN.data.training.valid.sum())} · training queries/update {QUERY_COUNT} · {QUERY_MODE} · independent queries {int(DIAGNOSTIC_PLAN.data.diagnostic.valid.sum())} · requested updates {DIAGNOSTIC_UPDATES} · optimizer updates so far **0**"))
    display(HTML("<details><summary>Complete resolved experiment, optimizer, objective and identities — review before launch</summary><pre>"+html.escape(json.dumps(resolved,indent=2,default=str))+"</pre></details>"))
    from pdeep.learning_diagnostics.plotting import plots
    plots(DIAGNOSTIC_PLAN.models[0], DIAGNOSTIC_PLAN.data,
        Path(SETUP["local_output_dir"])/"learning_diagnostics/previews"/DIAGNOSTIC_REVIEWED_IDENTITY[:16], label=DIAGNOSTIC_MODEL+" initial")
    import matplotlib.pyplot as plt
    plt.show()
else:
    print("Diagnostic model/data preparation blocked by missing/incompatible numerical certification.")

In [ ]:
# Optional non-optimizing preflight / inspection of the ACTUAL old trained weights.
if RUN_DIAGNOSTIC_PREFLIGHT and DIAGNOSTIC_PLAN is not None:
    DIAGNOSTIC_PREFLIGHT = preflight(DIAGNOSTIC_PLAN, loss_gradients=DIAGNOSTIC_LOSS_GRADIENTS, observations=True)
    from pdeep.artifact_io import atomic_json
    PREFLIGHT_PATH = Path(SETUP["local_output_dir"])/"learning_diagnostics/preflight"/(DIAGNOSTIC_PLAN.identity+".json")
    atomic_json(PREFLIGHT_PATH, DIAGNOSTIC_PREFLIGHT)
    print("Non-optimizing diagnostic report:", PREFLIGHT_PATH)
if INSPECT_CHECKPOINT is not None:
    CERTIFICATION.require()
    from pdeep.learning_diagnostics.inspection import inspect_mixed_checkpoint
    INSPECTION, INSPECTION_PATH = inspect_mixed_checkpoint(INSPECT_CHECKPOINT, CERTIFICATE,
        SETUP["local_output_dir"], device=DIAGNOSTIC_PLAN.configs[0].device if DIAGNOSTIC_PLAN else DIAGNOSTIC_DEVICE,
        gradients=DIAGNOSTIC_LOSS_GRADIENTS)
    display(Markdown(f"Read-only trained-checkpoint report: `{INSPECTION_PATH}`. No optimizer constructed or updated."))
if RUN_STRIPING_AUDIT:
    if STRIPING_CHECKPOINT is None:
        raise ValueError("Select a verified trained checkpoint manifest, keeping its .pt payload beside it.")
    from pdeep.learning_diagnostics.inspection import load_model
    from pdeep.learning_diagnostics.striping import audit_striping
    from pdeep.learning_diagnostics.data import prepare_data
    trained, stripe_metadata = load_model(Path(STRIPING_CHECKPOINT).expanduser(), CERTIFICATE,
        device=DIAGNOSTIC_PLAN.configs[0].device)
    saved = stripe_metadata["config"]
    stripe_data = prepare_data(Path(SETUP["local_output_dir"])/"learning_diagnostics/stripe_data", CERTIFICATE,
        experiment=saved["experiment"], seed=saved["root_seed"], count=saved["query_count"],
        conditioning_h=saved["conditioning_h"])
    STRIPING_REPORT = audit_striping(trained, stripe_data,
        Path(SETUP["local_output_dir"])/"learning_diagnostics/striping"/stripe_metadata["artifact_id"])
    display(STRIPING_REPORT["invariance_max_standardized_errors"])
    display({k:v["metrics"]["standardized_MSE"] for k,v in STRIPING_REPORT["layouts"].items()})
    print("Saved raw-coordinate metrics and scatter figures; no plot interpolation or optimizer update.")


### Execute the selected diagnostic
Set `DIAGNOSTIC_RUN_MODE="train"` (fresh) or `"resume"` with matching action/checkpoint. Rerun configuration and review, then execute the next cell. It starts immediately and stays executing until completion/error. **Run All with an enabled mode can start it too.** No other diagnostic is selected automatically; the paired C ablation intentionally runs its two objectives.

The one dashboard shows structured events; complete JSONL, traceback reports, fixed-data identities and immutable checkpoints remain on disk. Weak/initial/boundary losses are model-specific diagnostics, separate from reconstruction. Wave R2 is only an element-interior derivative diagnostic for the C0/H1 geometry path, never a global strong PDE score.

In [ ]:
DIAGNOSTIC_RESULTS = {}
if DIAGNOSTIC_RUN_MODE != "preview":
    if diagnostic_settings() != DIAGNOSTIC_SETTINGS:
        raise RuntimeError("Diagnostic configuration changed; rerun preparation and review.")
    CERTIFICATION.require()
    if DIAGNOSTIC_PLAN is None:
        raise RuntimeError("Prepare and review the diagnostic first.")
    DIAGNOSTIC_MONITOR = Dashboard(notebook=True)
    DIAGNOSTIC_RESULTS = launch_diagnostic(DIAGNOSTIC_PLAN,
        expected_identity=DIAGNOSTIC_REVIEWED_IDENTITY,
        run_mode=DIAGNOSTIC_RUN_MODE, monitor=DIAGNOSTIC_MONITOR)
else:
    print("Diagnostic preview: no optimizer constructed or updated. Manual diagnostic training pending.")

In [ ]:
from pdeep.learning_diagnostics.plotting import final_results
if DIAGNOSTIC_RESULTS:
    final_results(DIAGNOSTIC_RESULTS)
    import matplotlib.pyplot as plt
    plt.show()
else:
    print("Learning diagnosis: pending manual experiments. Initial/untrained plots do not establish learning or correct conditioning.")

## Optional matched query-strategy comparison
Planning performs no training. Select **one budget**: equal updates, or equal supervised field-query evaluations. Default1000 updates means the equal-query budget is1,024,000 (1000/1000/250/125 updates). Equal queries still differ in context/weak-physics work and wall/GPU time. Every arm reloads the one saved initial model, uses the identical physical case/sensors/reference/fixed-control/evaluation sets, and saves separately. No warm starts across arms.

Keep this disabled unless deliberately preparing a comparison. Setting comparison mode to train and executing its launch cell synchronously runs the selected arms; **Run All with an enabled mode can train**. Individual saved arms resume via the diagnostic checkpoint controls.

In [ ]:
ENABLE_QUERY_COMPARISON = False
COMPARISON_AMP_MODE = "off"  # One common FP32 control; explicit bf16 only after all GPU pilots pass.
COMPARISON_BUDGET = "equal_updates"  # Or equal_queries; budget must be divisible by8192.
TOTAL_QUERY_BUDGET = 1024000
COMPARISON_ARMS = ("fixed_1024", "resampled_1024", "dense_resampled_4096", "dense_resampled_8192")
COMPARISON_RUN_MODE = "preview"
COMPARISON_PLAN = None
if ENABLE_QUERY_COMPARISON:
    from pdeep.learning_diagnostics.comparison import prepare_comparison, launch_comparison
    common = diagnostic_settings().copy()
    for key in ("query_mode","query_count","checkpoints"):
        common.pop(key, None)
    common["amp_mode"] = COMPARISON_AMP_MODE
    COMPARISON_PLAN = prepare_comparison(**common, budget=COMPARISON_BUDGET,
        total_queries=TOTAL_QUERY_BUDGET)
    COMPARISON_REVIEWED_IDENTITY = COMPARISON_PLAN.identity
    display(COMPARISON_PLAN.preview())

In [ ]:
COMPARISON_RESULTS = {}
if COMPARISON_RUN_MODE != "preview":
    if not ENABLE_QUERY_COMPARISON or COMPARISON_PLAN is None:
        raise RuntimeError("Enable and review one comparison first.")
    if diagnostic_settings() != DIAGNOSTIC_SETTINGS:
        raise RuntimeError("Settings changed; rerun preparation/review.")
    COMPARISON_RESULTS = launch_comparison(COMPARISON_PLAN,
        expected_identity=COMPARISON_REVIEWED_IDENTITY, run_mode=COMPARISON_RUN_MODE,
        selected_arms=COMPARISON_ARMS, monitor=Dashboard(notebook=True))
    final_results(COMPARISON_RESULTS)
else:
    print("Comparison preview: no arm training launched.")

## Optional mixed / bounded training workflow
This separate section preserves the original training APIs/objectives. `tiny_overfit` samples six fixed TRAIN cases with resampled queries: it is **not strict fixed-query memorization**. `bounded` uses paired unseen-case validation and the declared lazy train pool. Leave its run mode at preview when executing the single-case diagnostics above.

In [ ]:
BENCHMARK = "G1_smooth_v1"
MODELS = ("A_GEO", "C_GEO")
CAPACITY = "medium"
PDE_FAMILIES = ("poisson", "heat", "wave")
DEVICE = "auto"  # Colab auto requires validated CUDA; local auto uses available MPS/CPU
EXPERIMENT = "tiny_overfit"  # bounded/smoke also available
OVERFIT_UPDATES = 100
UPDATES = 1000  # bounded budget; tiny_overfit uses OVERFIT_UPDATES automatically; smoke: <=3
ACTION = "fresh"  # fresh or resume
RESUME_CHECKPOINT = None  # verified Drive/local smooth-update-XXXXXXXX.json
SAVE_EVERY = 250
VALIDATE_EVERY = 250
RUN_MODE = "preview"  # preview, train, resume; executing launch in enabled mode starts immediately


In [ ]:
# Optional advanced settings; defaults keep the existing architecture-specific loss.
ADVANCED = {}  # e.g. {"cases_per_geometry":4}; finite query bank is explicit
RUN_NAME = None  # unique run tag; existing fresh destinations are rejected
PERSISTENCE = SETUP["persistence"]


In [ ]:
from pdeep.smooth.workflows import prepare, preview
from pdeep.smooth.certification import load_record
from pdeep.smooth.definition import manifest
from IPython.display import display, Markdown
import json

def training_settings():
    if SETUP_CONFIGURATION != (REPO_URL, REPO_REF, PROJECT_DIR, USE_GOOGLE_DRIVE, DRIVE_ROOT, LOCAL_OUTPUT_DIR):
        raise RuntimeError("Setup changed: rerun setup and preview.")
    device = "cuda" if IS_COLAB and DEVICE == "auto" else DEVICE
    if IS_COLAB and not str(device).startswith("cuda"):
        raise RuntimeError("Colab requires validated CUDA; no silent CPU fallback.")
    return dict(models=MODELS, benchmark=BENCHMARK, capacity=CAPACITY,
        pdes=PDE_FAMILIES, device=device, experiment=EXPERIMENT, updates=OVERFIT_UPDATES if EXPERIMENT == "tiny_overfit" else UPDATES,
        action=ACTION, resume_checkpoint=RESUME_CHECKPOINT,
        save_every=SAVE_EVERY, validate_every=VALIDATE_EVERY,
        output_dir=SETUP["local_output_dir"], persistence=PERSISTENCE,
        run_name=RUN_NAME, advanced=ADVANCED.copy())

SETTINGS = {**training_settings(), "certificate_path": CERTIFICATION.location if CERTIFICATION.verified else None}


In [ ]:
PLAN = None
if CERTIFICATION.verified:
    PLAN = prepare(**SETTINGS)
    REVIEWED_IDENTITY = PLAN.identity
    REVIEW = preview(PLAN)
    display(Markdown("|Model|Parameters|Device|Experiment|Updates|Cases|\n|---|---:|---|---|---:|---:|\n"+
        "\n".join(f"|{r['architecture']}|{r['parameter_count']}|{r['device']}|{r['experiment']}|{r['successful_updates']}|{r['case_pool']}|" for r in REVIEW)))
    display(HTML("<details><summary>Full mixed/bounded plan and optimizer settings</summary><pre>"+html.escape(json.dumps({"plan":REVIEW,"configs":[__import__('dataclasses').asdict(c) for c in PLAN.configs]},indent=2,default=str))+"</pre></details>"))
else:
    print("Mixed/bounded plan blocked by numerical certification. No automatic qualification.")

### Execute mixed/bounded training directly
Set `RUN_MODE="train"` with `ACTION="fresh"`, or `"resume"` with `ACTION="resume"` and the saved verified manifest. Rerun review and execute the next cell. **It starts immediately and executes synchronously. Run All with enabled training can also start it.** The same dashboard handle serves sequential A/C runs. No button or confirmation is required.

In [ ]:
from pdeep.smooth.workflows import launch
RESULTS = {}
if RUN_MODE != "preview":
    if {**training_settings(), "certificate_path": CERTIFICATION.location if CERTIFICATION.verified else None} != SETTINGS:
        raise RuntimeError("Configuration changed; rerun preview.")
    if RUN_MODE not in {"train", "resume"} or (RUN_MODE == "resume") != (ACTION == "resume"):
        raise RuntimeError("Choose consistent train/fresh or resume/resume action.")
    CERTIFICATION.require()
    if PLAN is None:
        raise RuntimeError("Prepare/review a compatible plan first.")
    LIVE_MONITOR = Dashboard(notebook=True)
    RESULTS = launch(PLAN, expected_identity=REVIEWED_IDENTITY, notebook=True, monitor=LIVE_MONITOR)
else:
    print("Mixed/bounded preview: zero optimizer updates.")

In [ ]:
from pdeep.smooth.plotting import validation_table, history_plots, time_plot, representative_field
if not RESULTS:
    print("Mixed/bounded learned accuracy: pending user execution.")
for name, engine in RESULTS.items():
    display(Markdown(f"**{name}** · updates {engine.state.successful} · failed {engine.state.failed} · skipped {engine.state.skipped} · Drive **{engine.state.persistence.get('Drive_status')}**\n\nCheckpoint: `{engine.state.last_checkpoint}`\n\nReports: `{engine.run_dir}`"))
    if engine.config.preset == "tiny_overfit": display(engine.overfit_assessment())
    if engine.latest_validation: display(Markdown(validation_table(engine.latest_validation)))
    history_plots(engine.run_dir)
    representative_field(engine, pde="wave")
import matplotlib.pyplot as plt
plt.show()

## Resume, persistence and interpretation
Diagnostic checkpoints are under `SETUP["local_output_dir"]/learning_diagnostics/<experiment>/<objective>/G1_smooth_v1/runs/<model>/<run_id>/checkpoints/diagnostic-update-XXXXXXXX.json`; inference exports, figures, JSONL and diagnostics sit alongside. Existing mixed runs remain in `G1_smooth_v1/runs`. Both use the verified local-first checkpoint copier at periodic saves. Drive failures retain valid local checkpoints and show a warning/status; Colab local storage is temporary.

After disconnect, open this notebook from GitHub at the same commit/tag, run setup and remount Drive. Select the exact diagnostic model/experiment/capacity/seed/budget, set `DIAGNOSTIC_ACTION="resume"`, choose its verified Drive manifest in advanced `DIAGNOSTIC_CHECKPOINTS`, review, set `DIAGNOSTIC_RUN_MODE="resume"`, and execute. Saved optimizer/scheduler/RNG/fixed queries/counters are restored. Budgets are not silently extended, and mixed checkpoints are never implicit single-case warm starts. A paired ablation requires both manifests keyed FULL/FIELD_ONLY. Use `INSPECT_CHECKPOINT` for read-only analysis of an old mixed run, or select it as the observation-sensitivity checkpoint with fresh action (read-only, not training).

Field/auxiliary/BC/residual errors are separate. Gradients establish reachability, not reliable conditioning. The sensitivity experiment also compares two genuine hidden states against their own truths and empty/zero-prior/shuffled controls. A passed memorization threshold is not generalization evidence. See `docs/geometry_learning_diagnostics.md` for experiment selections and diagnosis guide. Test split remains untouched; original T1 and archival APIs remain available.